In [5]:
# This chunk load Packages needed
import xarray as xr # for xarray data 
import matplotlib.pyplot as plt # for plotting
import numpy as np # for arrays
import pandas as pd 
from pathlib import Path
import fsspec
import os
from glob import glob

In [2]:
# Process the all the ssp245 data into three 10 year chunks (1850s, 2000s, 2090s) and take averages across the decades 
# keep monthly level data, group into one file and output

# load files
ssp245_locs = ["gs://leap-persistent/abbysh/phytoplankton/cmip6_wcarbonatecalcs_1850-2100/ssp245/ensemble_average/ssp245_ensemble_average.zarr",
        "gs://leap-persistent/abbysh/phytoplankton/cmip6_wcarbonatecalcs_1850-2100/ssp245/CESM2/model_average/CESM2_average.zarr",
        "gs://leap-persistent/abbysh/phytoplankton/cmip6_wcarbonatecalcs_1850-2100/ssp245/CESM2-WACCM/model_average/CESM2-WACCM_average.zarr",
        "gs://leap-persistent/abbysh/phytoplankton/cmip6_wcarbonatecalcs_1850-2100/ssp245/CanESM5/model_average/CanESM5_average.zarr",
        "gs://leap-persistent/abbysh/phytoplankton/cmip6_wcarbonatecalcs_1850-2100/ssp245/CanESM5-CanOE/model_average/CanESM5-CanOE_average.zarr",
        "gs://leap-persistent/abbysh/phytoplankton/cmip6_wcarbonatecalcs_1850-2100/ssp245/GFDL-ESM4/model_average/GFDL-ESM4_average.zarr"
       ]

# Output path
output_dir = Path("decade_averages/ssp245_monthly/")
output_dir.mkdir(parents=True, exist_ok=True)

# Define target decades and compute target years
target_decades = [1850, 2000, 2090]
target_years = [year for decade in target_decades for year in range(decade, decade + 10)]

for zarr_path in ssp245_locs:
    print(f"Processing: {zarr_path}")
    model_name = Path(zarr_path).stem

    # Load dataset
    ds = xr.open_zarr(zarr_path, consolidated=True)

    # Filter to target years
    ds = ds.where(ds.time.dt.year.isin(target_years), drop=True)

    # Add 'month' and 'decade' as coordinates (fix: use .data to avoid ambiguity)
    ds = ds.assign_coords(
        month=("time", ds.time.dt.month.data),
        decade=("time", ((ds.time.dt.year // 10 * 10).astype(str) + "s").data)
    )

    # Group by (decade, month)
    ds_mean = ds.groupby(["decade", "month"]).mean(dim="time")
    ds_std = ds.groupby(["decade", "month"]).std(dim="time")

    # Rename std variables
    ds_std = ds_std.rename({var: f"{var}_std" for var in ds_std.data_vars})

    # Merge and save
    ds_combined = xr.merge([ds_mean, ds_std])
    output_filename = output_dir / f"{model_name}_monthly_decadal.nc"
    ds_combined.to_netcdf(output_filename)

    print(f"Saved to: {output_filename}")

NameError: name 'Path' is not defined

In [1]:
# Process the all the ssp585 data into three 10 year chunks (1850s, 2000s, 2090s) and take averages across the decades 
# keep monthly level data, group into one file and output

# load files
ssp585_locs = ["gs://leap-persistent/abbysh/phytoplankton/cmip6_wcarbonatecalcs_1850-2100/ssp585/ensemble_average/ssp585_ensemble_average.zarr",
        "gs://leap-persistent/abbysh/phytoplankton/cmip6_wcarbonatecalcs_1850-2100/ssp585/CESM2/model_average/CESM2_average.zarr",
        "gs://leap-persistent/abbysh/phytoplankton/cmip6_wcarbonatecalcs_1850-2100/ssp585/CESM2-WACCM/model_average/CESM2-WACCM_average.zarr",
        "gs://leap-persistent/abbysh/phytoplankton/cmip6_wcarbonatecalcs_1850-2100/ssp585/CanESM5/model_average/CanESM5_average.zarr",
        "gs://leap-persistent/abbysh/phytoplankton/cmip6_wcarbonatecalcs_1850-2100/ssp585/CanESM5-CanOE/model_average/CanESM5-CanOE_average.zarr",
        "gs://leap-persistent/abbysh/phytoplankton/cmip6_wcarbonatecalcs_1850-2100/ssp585/GFDL-ESM4/model_average/GFDL-ESM4_average.zarr"
       ]

# Output path
output_dir = Path("decade_averages/ssp585_monthly/")
output_dir.mkdir(parents=True, exist_ok=True)

# Define target decades and compute target years
target_decades = [1850, 2000, 2090]
target_years = [year for decade in target_decades for year in range(decade, decade + 10)]

for zarr_path in ssp585_locs:
    print(f"Processing: {zarr_path}")
    model_name = Path(zarr_path).stem

    # Load dataset
    ds = xr.open_zarr(zarr_path, consolidated=True)

    # Filter to target years
    ds = ds.where(ds.time.dt.year.isin(target_years), drop=True)

    # Add 'month' and 'decade' as coordinates (fix: use .data to avoid ambiguity)
    ds = ds.assign_coords(
        month=("time", ds.time.dt.month.data),
        decade=("time", ((ds.time.dt.year // 10 * 10).astype(str) + "s").data)
    )

    # Group by (decade, month)
    ds_mean = ds.groupby(["decade", "month"]).mean(dim="time")
    ds_std = ds.groupby(["decade", "month"]).std(dim="time")

    # Rename std variables
    ds_std = ds_std.rename({var: f"{var}_std" for var in ds_std.data_vars})

    # Merge and save
    ds_combined = xr.merge([ds_mean, ds_std])
    output_filename = output_dir / f"{model_name}_monthly_decadal.nc"
    ds_combined.to_netcdf(output_filename)

    print(f"Saved to: {output_filename}")

NameError: name 'Path' is not defined

In [9]:
# Group all variables into one .nc file for ease of use
# Set your input folder here
base_dir = "decade_averages/ssp585_monthly"  # <--- ran on both folders, decade_averages/ssp585_monthly and decade_averages/ssp245_monthly

# Variables to extract
vars_to_keep = ["aqueous_CO2", "bicarbonate", "carbonate", "tos", "pH", "dic", "pCO2"]

# Create output directory inside base_dir
out_dir = os.path.join(base_dir, "timeseries")
os.makedirs(out_dir, exist_ok=True)

# List all .nc files except ensemble
nc_files = glob(os.path.join(base_dir, "*.nc"))
nc_files = [f for f in nc_files if "ensemble" not in f.lower()]

for in_path in nc_files:
    print(f"Processing: {in_path}")

    try:
        ds = xr.open_dataset(in_path)
        
        # Check if both 'decade' and 'month' are in dataset
        if not {"decade", "month"}.issubset(ds.coords):
            print(f"⚠️ Skipping {in_path} because 'decade' or 'month' coordinates not found.")
            continue

        decades = ds["decade"].values
        months = ds["month"].values

        # Create datetime index for all combinations
        times = []
        for d in decades:
            year_str = str(d)[:4]
            for m in months:
                times.append(pd.Timestamp(f"{year_str}-{int(m):02d}-01"))
        times = pd.to_datetime(times)

        # Stack decade and month into single dimension 'time'
        ds_stacked = ds.stack(time=("decade", "month"))

        # Drop existing 'time' coordinate and also 'decade' and 'month' variables before assigning new time
        ds_stacked = ds_stacked.drop_vars(["time", "decade", "month"], errors='ignore')

        # Now assign new time coordinate
        ds_stacked = ds_stacked.assign_coords(time=times)

        ds_sel = ds_stacked[vars_to_keep]

        # Transpose dims so time is first, then latitude and longitude
        ds_sel = ds_sel.transpose("time", "latitude", "longitude")

        for var in ds_sel.data_vars:
            out_path = os.path.join(out_dir, f"{var}_timeseries_{os.path.basename(in_path)}")
            ds_sel[[var]].to_netcdf(out_path)
            print(f"Saved: {out_path}")

    except Exception as e:
        print(f"⚠️ Error processing {in_path}: {e}")


Processing: decade_averages/ssp585_monthly/CESM2_average_monthly_decadal.nc
Saved: decade_averages/ssp585_monthly/timeseries/aqueous_CO2_timeseries_CESM2_average_monthly_decadal.nc
Saved: decade_averages/ssp585_monthly/timeseries/bicarbonate_timeseries_CESM2_average_monthly_decadal.nc
Saved: decade_averages/ssp585_monthly/timeseries/carbonate_timeseries_CESM2_average_monthly_decadal.nc
Saved: decade_averages/ssp585_monthly/timeseries/tos_timeseries_CESM2_average_monthly_decadal.nc
Saved: decade_averages/ssp585_monthly/timeseries/pH_timeseries_CESM2_average_monthly_decadal.nc
Saved: decade_averages/ssp585_monthly/timeseries/dic_timeseries_CESM2_average_monthly_decadal.nc
Saved: decade_averages/ssp585_monthly/timeseries/pCO2_timeseries_CESM2_average_monthly_decadal.nc
Processing: decade_averages/ssp585_monthly/CESM2-WACCM_average_monthly_decadal.nc
Saved: decade_averages/ssp585_monthly/timeseries/aqueous_CO2_timeseries_CESM2-WACCM_average_monthly_decadal.nc
Saved: decade_averages/ssp585_